# Graph Benchmark — standalone

Compares an arbitrary predicted graph against a ground-truth `ClusteredGraph`,
using **Capped Semantic Fβ with one-to-one optimal matching**.

Per side (nodes / edges):
1. embed every item with context-aware embeddings;
2. compute cosine similarity matrix;
3. cap below `tau`:  `q[i,j] = max(0, (sim - tau) / (1 - tau))`;
4. find one-to-one matching that maximizes Σq (Hungarian);
5. `precision = Σq / N_pred`,  `recall = Σq / N_gt`,  `Fβ` aggregates them.

`GraphScore = node_weight·NodeFβ + edge_weight·EdgeFβ`. Defaults: `tau=0.6`,
`beta=1.0`, `node_weight=0.6`, `edge_weight=0.4`.

Supports both `clustered_graph.json` (single) and
`multi_clustered_graph.json` (sweep across methods/parameters) — the type is
auto-detected from the file.

In [1]:
import sys
from pathlib import Path

# make `llm_v2` importable from anywhere this notebook lives
_root = Path().resolve()
for cand in (_root, *_root.parents):
    if (cand / 'llm_v2').exists():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break

from llm_v2.config_schema import EmbeddingConfig
from llm_v2.models.embedder import Embedder
from llm_v2.schemas.clustered_graph import ClusteredGraph, MultiClusteredGraph
from llm_v2.benchmark import (
    evaluate,
    evaluate_graph,
    evaluate_multi_graph,
    load_graph_auto,
    print_metrics,
    print_multi_metrics,
    best_variant,
    show_node_alignments,
    show_edge_alignments,
    multi_metrics_to_dict,
)

/home/platoon/graph/graph_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inputs

In [9]:
# point to either a single ClusteredGraph json or a MultiClusteredGraph json —
# auto-detected by the loader.
PRED_GRAPH = Path('/home/platoon/graph/semantic-graph/exps/baseline/output/clustered_graph.json')
# PRED_GRAPH = Path('../llm_v2/output/multi_clustered_graph.json')

GT_GRAPH = Path('final_bench/graph_clustered.json')

# embedding context — coreference-resolved text matches what the predicted
# graph saw; falls back to the source text
SOURCE_TEXT_PATH   = Path('final_bench/formated_fragment2.md')
RESOLVED_TEXT_PATH = Path('../llm_v2/output/coreference_resolved.txtt')

# embedder
EMBED_MODEL  = 'all-mpnet-base-v2'
EMBED_DEVICE = 'cuda'

# metric knobs
TAU_NODE    = 0.6
TAU_EDGE    = 0.6
BETA        = 1.0
NODE_WEIGHT = 0.6
EDGE_WEIGHT = 0.4
NODE_WINDOW = 300
EDGE_WINDOW = 400
TOP_K       = 10

# where to dump JSON metrics
OUT_PATH = Path('benchmark_metrics.json')

In [10]:
pred = load_graph_auto(PRED_GRAPH)
gt   = ClusteredGraph(**__import__('json').loads(GT_GRAPH.read_text(encoding='utf-8')))

if RESOLVED_TEXT_PATH.exists():
    source_text = RESOLVED_TEXT_PATH.read_text(encoding='utf-8')
    text_provenance = f'resolved ({RESOLVED_TEXT_PATH})'
else:
    source_text = SOURCE_TEXT_PATH.read_text(encoding='utf-8')
    text_provenance = f'source   ({SOURCE_TEXT_PATH})'

is_multi = isinstance(pred, MultiClusteredGraph)
if is_multi:
    n_variants = sum(len(mr.graphs) for mr in pred.methods.values())
    methods_summary = ', '.join(f'{m}={len(mr.param_labels)}' for m, mr in pred.methods.items())
    print(f'Pred : MultiClusteredGraph — {n_variants} variants ({methods_summary})  ({PRED_GRAPH})')
else:
    print(f'Pred : ClusteredGraph — {len(pred.nodes)} nodes, {len(pred.edges)} edges  ({PRED_GRAPH})')
print(f'GT   : {len(gt.nodes)} nodes, {len(gt.edges)} edges  ({GT_GRAPH})')
print(f'Text : {len(source_text)} chars — {text_provenance}')

Pred : ClusteredGraph — 152 nodes, 112 edges  (/home/platoon/graph/semantic-graph/exps/baseline/output/clustered_graph.json)
GT   : 55 nodes, 51 edges  (final_bench/graph_clustered.json)
Text : 15466 chars — source   (final_bench/formated_fragment2.md)


In [11]:
embedder = Embedder(EmbeddingConfig(model_name=EMBED_MODEL, device=EMBED_DEVICE))
print(f'Embedder loaded: {EMBED_MODEL} (dim={embedder.dim})')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1171.05it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedder loaded: all-mpnet-base-v2 (dim=768)


## Metrics

The next cell branches on `is_multi`:
- **single graph** → one `GraphMetrics` printed via `print_metrics`;
- **multi graph** → comparison table via `print_multi_metrics`, then drill-down on the best variant by `graph_score`.

In [5]:
common_kwargs = dict(
    tau_node=TAU_NODE,
    tau_edge=TAU_EDGE,
    beta=BETA,
    node_weight=NODE_WEIGHT,
    edge_weight=EDGE_WEIGHT,
    node_window=NODE_WINDOW,
    edge_window=EDGE_WINDOW,
)

if is_multi:
    multi_metrics = evaluate_multi_graph(
        multi=pred, gt=gt, source_text=source_text, embedder=embedder,
        **common_kwargs,
    )
    print_multi_metrics(multi_metrics, sort_by='graph_score')

    # pick best variant for alignment inspection below
    best_method, best_param, best_m = best_variant(multi_metrics, by='graph_score')
    pred_for_alignments = pred.methods[best_method].graphs[best_param]
    metrics_for_alignments = best_m
    print(f'\nBest variant by GraphScore: method={best_method}, param={best_param}')
    print_metrics(best_m)
else:
    metrics = evaluate_graph(
        pred=pred, gt=gt, source_text=source_text, embedder=embedder,
        **common_kwargs,
    )
    print_metrics(metrics)
    pred_for_alignments = pred
    metrics_for_alignments = metrics
    multi_metrics = None

GraphScore = 0.6·NodeF1 + 0.4·EdgeF1  =  0.1797

Nodes (pred=152, gt=55, matched=35, tau=0.6, beta=1):
  TP(soft)  = 19.6972
  precision = 0.1296
  recall    = 0.3581
  F1        = 0.1903
Edges (pred=112, gt=51, matched=26, tau=0.6, beta=1):
  TP(soft)  = 13.3527
  precision = 0.1192
  recall    = 0.2618
  F1        = 0.1638


## Alignments — best/worst matched pairs and unmatched items

For multi graphs, this block inspects the **best variant** picked above.
Override `pred_for_alignments` and `metrics_for_alignments` to look at any other variant.

In [6]:
show_node_alignments(pred_for_alignments, gt, metrics_for_alignments, top_k=TOP_K)

Matched node pairs: 35 / min(152, 55)=55

Top 10 matched (by quality q):
  [q=1.000]  'минимальный отступ'  ↔  'минимальный отступ'
  [q=1.000]  'гиперплоскость'  ↔  'гиперплоскость'
  [q=1.000]  'выборка'  ↔  'выборка'
  [q=1.000]  'регрессия'  ↔  'регрессия'
  [q=0.988]  'разделять поверхность'  ↔  'разделяющая поверхность'
  [q=0.967]  'задача классификация'  ↔  'задача классификации'
  [q=0.961]  'ошибка перцептрон'  ↔  'ошибка перцептрона'
  [q=0.931]  'реализация случайный величина'  ↔  'логистическая регрессия'
  [q=0.892]  'линейно разделимый'  ↔  'линейная разделимость'
  [q=0.873]  'то'  ↔  'MSE'

Bottom 10 matched:
  [q=0.259]  'из желание посмотреть на классификация как на задача предсказание вероятность'  ↔  'классификация на $K$ классов'
  [q=0.245]  'наш таргет'  ↔  'таргет $y$'
  [q=0.218]  'отступ ( англ . * margin * ) классификатор'  ↔  'отрицательный отступ'
  [q=0.187]  'попробовать предсказывать число $ -1 $ и $ 1 $ , минимизировать для это , например , mse с после

In [7]:
show_edge_alignments(pred_for_alignments, gt, metrics_for_alignments, top_k=TOP_K)

Matched edge pairs: 26 / min(112, 51)=51

Top 10 matched (by quality q):
  [q=0.903]  для который правдоподобие максимально —[интересовать]→ мы
           ↔  регрессия —[минимизирует]→ MSE
  [q=0.858]  регрессия —[штрафовать]→ за ошибка на объект , который лежать близко к разделять плоскость , но не с сторона
           ↔  регрессия —[может быть наивным подходом к]→ задача классификации
  [q=0.811]  клик —[являться]→ на каждый обучать пример
           ↔  регрессия —[является плохим подходом для]→ задача классификации
  [q=0.775]  выборка —[называться]→ линейно разделимый
           ↔  выборка —[может обладать свойством]→ линейная разделимость
  [q=0.742]  мы —[заменить]→ тут всё в порядок
           ↔  функционал ошибки классификатора —[минимизирует]→ число ошибок классификатора
  [q=0.716]  мы —[увидеть]→ в параграф
           ↔  логистическая регрессия —[задаёт]→ разделяющая поверхность
  [q=0.704]  понять , насколько вероятно получить дать значение таргет $ y$ при данные $ x$ и вес

## Save

In [8]:
import json

if is_multi:
    payload = multi_metrics_to_dict(multi_metrics)
    payload['_best'] = {
        'method': best_method,
        'param':  best_param,
        **best_m.summary(),
    }
else:
    payload = metrics.summary()

OUT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Saved {OUT_PATH.resolve()}')

Saved /home/platoon/graph/semantic-graph/benchmark/benchmark_metrics.json
